# 🤗 RAG Pipeline — HuggingFace + Multi-Document Support
> Supports **PDF**, **Excel (.xlsx)**, and **Word (.docx)** uploads  
> Uses **HuggingFace** Embeddings + Inference API (replaces OpenAI)

### RAG Flow
```
Upload Docs → Split → Embed (HF) → FAISS Store → Retrieve → Augment → Generate (HF LLM)
```

## 🔑 Step 0 — Set HuggingFace API Key

In [17]:
import os

# ✅ Replace with your OpenRouter API key
# Get it from: https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = "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"

print("✅ OpenRouter API key set!")


✅ OpenRouter API key set!


## 📦 Install Libraries

In [18]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    huggingface-hub \
    pypdf \
    openpyxl \
    python-docx \
    python-dotenv

print("✅ All libraries installed!")

✅ All libraries installed!


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [20]:
pip install --upgrade langchain langchain-text-splitters langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.6
    Uninstalling langgraph-prebuilt-1.0.6:
      Successfully uninstalled langgraph-prebuilt-1.0.6
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.6
    Uninstalling langgraph-1.0.6:
      Successfully uninstalled langgraph-1.0.6
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.4
    Uninstalling langchain-1.2.4:
      Successfully uninstalled langchain-1.2.4


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceEndpoint


# Rest of your code


## 📥 Imports

In [22]:
import os
import tempfile
import pandas as pd

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Document
from langchain_core.documents import Document

# Embeddings and LLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceEndpoint

# Vector Store
from langchain_community.vectorstores import FAISS

# Prompts and Runnables
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Document Loaders
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

print("✅ Imports done!")


✅ Imports done!


## 📂 Step 1a — Document Ingestion
> Upload your **PDF**, **Excel**, or **Word** files here.  
> In Google Colab: use the 📁 sidebar to upload, then provide the file path below.

In [24]:
# ─────────────────────────────────────────────────────────────
# 📌 CONFIGURE: Add your file paths here
# In Colab: upload files via the Files panel (left sidebar)
# Then update the paths below
# ─────────────────────────────────────────────────────────────

FILE_PATHS = [

     "/content/AI OPS.pdf",
    # "sample.xlsx",
    # "sample.docx",
]

# ─────────────────────────────────────────────────────────────
# Document loader — handles PDF, Excel, Word automatically
# ─────────────────────────────────────────────────────────────

def load_documents(file_paths: list) -> list:
    """Load PDF, Excel (.xlsx), and Word (.docx) files into LangChain Documents."""
    all_docs = []

    for path in file_paths:
        ext = os.path.splitext(path)[-1].lower()
        print(f"📄 Loading: {path}  [{ext}]")

        if ext == ".pdf":
            loader = PyPDFLoader(path)
            docs = loader.load()
            all_docs.extend(docs)
            print(f"   ✅ Loaded {len(docs)} pages from PDF")

        elif ext in [".xlsx", ".xls"]:
            # Read all sheets
            xls = pd.read_excel(path, sheet_name=None)
            for sheet_name, df in xls.items():
                text = f"Sheet: {sheet_name}\n" + df.to_string(index=False)
                doc = Document(
                    page_content=text,
                    metadata={"source": path, "sheet": sheet_name}
                )
                all_docs.append(doc)
            print(f"   ✅ Loaded {len(xls)} sheet(s) from Excel")

        elif ext == ".docx":
            loader = Docx2txtLoader(path)
            docs = loader.load()
            all_docs.extend(docs)
            print(f"   ✅ Loaded {len(docs)} section(s) from Word")

        else:
            print(f"   ⚠️  Unsupported format: {ext} — skipping")

    print(f"\n📦 Total documents loaded: {len(all_docs)}")
    return all_docs


raw_docs = load_documents(FILE_PATHS)

# Preview first doc
if raw_docs:
    print("\n--- Preview of first document ---")
    print(raw_docs[0].page_content[:500])
    print("Metadata:", raw_docs[0].metadata)
else:
    print("⚠️  No files loaded. Add file paths to FILE_PATHS above and re-run.")

📄 Loading: /content/AI OPS.pdf  [.pdf]
   ✅ Loaded 535 pages from PDF

📦 Total documents loaded: 535

--- Preview of first document ---
Chip Huyen
 AI Engineering
Building Applications  
with Foundation Models
Metadata: {'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '/content/AI OPS.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}


## ✂️ Step 1b — Text Splitting

In [25]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(raw_docs)

print(f"✅ Total chunks created: {len(chunks)}")

# Preview a chunk
if chunks:
    print("\n--- Sample Chunk ---")
    print(chunks[0].page_content)
    print("Metadata:", chunks[0].metadata)

✅ Total chunks created: 1506

--- Sample Chunk ---
Chip Huyen
 AI Engineering
Building Applications  
with Foundation Models
Metadata: {'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '/content/AI OPS.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}


## 🔢 Step 1c & 1d — Embedding + Vector Store
> Using **`sentence-transformers/all-MiniLM-L6-v2`** — free, runs locally, no API key needed for embeddings.

In [26]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Use a free embedding model
embedding_model_name = "BAAI/bge-small-en-v1.5"  # or any other free model

print("⏳ Loading embedding model (downloads once)...")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cpu"},  # Use "cuda" if GPU is available
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded!")

# Build FAISS vector store
print("⏳ Building FAISS vector store...")
vector_store = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store built! Total vectors: {vector_store.index.ntotal}")


⏳ Loading embedding model (downloads once)...


/tmp/ipython-input-3248567092.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
⏳ Building FAISS vector store...
✅ Vector store built! Total vectors: 1506


In [27]:
# Optional: Save and reload vector store (useful for large docs)
# vector_store.save_local("faiss_index")
# vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
print("Vector store ready. Uncomment above lines to persist to disk.")

Vector store ready. Uncomment above lines to persist to disk.


## 🔍 Step 2 — Retriever

In [28]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # retrieve top-4 most relevant chunks
)

print("✅ Retriever ready!")
print(retriever)

✅ Retriever ready!
tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d1191fa8440> search_kwargs={'k': 4}


In [29]:
# 🧪 Test the retriever
test_query = "What is this document about?"
test_results = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(test_results)} chunks:\n")
for i, doc in enumerate(test_results):
    print(f"[{i+1}] Source: {doc.metadata.get('source', 'N/A')} | Page: {doc.metadata.get('page', 'N/A')}")
    print(doc.page_content[:300])
    print("---")

Query: 'What is this document about?'
Retrieved 4 chunks:

[1] Source: /content/AI OPS.pdf | Page: 294
(Anthropic, 2024):
<document>
{{WHOLE_DOCUMENT}}
</document>
Here is the chunk we want to situate within the whole document:
<chunk>
{{CHUNK_CONTENT}}
</chunk>
Please give a short succinct context to situate this chunk within the
overall document for the purposes of improving search retrieval of the
---
[2] Source: /content/AI OPS.pdf | Page: 197
<<poem of joy>>.
Detectable format Choose from Answer with one of the following options: {options}.
Detectable format Minimum number
highlighted section
Highlight at least {N} sections in your answer with markdown, i.e.
*highlighted section*
Detectable format Multiple sections Your response must hav
---
[3] Source: /content/AI OPS.pdf | Page: 294
You can also augment each chunk with the questions it can answer. For customer
support, you can augment each article with related questions. For example, the article
on how to reset your password can

## 🤗 Step 3 — Augmentation (HuggingFace LLM)
> Using **Mistral-7B-Instruct** via HuggingFace Inference API (free tier).  
> You can swap `repo_id` with any other supported model.

In [30]:
!pip install langchain-openai

In [31]:
!pip show langchain-openai

Name: langchain-openai
Version: 1.1.12
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/openai
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, openai, tiktoken
Required-by: 


In [32]:
# ✅ Remove any HuggingFaceEndpoint — use this instead
from langchain_openai import ChatOpenAI
import os

os.environ["OPENROUTER_API_KEY"] = "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"

llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    temperature=0.2,
    max_tokens=512,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("✅ LLM ready!")

# Quick test
response = llm.invoke("Say hello in one sentence.")
print(response.content)

✅ LLM ready!
Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss!


In [33]:
!pip install -q langchain==0.3.25 langchain-core==0.3.58 langchain-community==0.3.24 langchain-openai langchain-huggingface==0.1.2 langchain-text-splitters==0.3.8 faiss-cpu sentence-transformers pypdf openpyxl python-docx docx2txt

ERROR: Cannot install langchain-community==0.3.24, langchain-core==0.3.58 and langchain==0.3.25 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [34]:
prompt = PromptTemplate(
    template="""<s>[INST]
You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information in the provided documents."
Be concise and clear.

Context:
{context}

Question: {question}
[/INST]""",
    input_variables=["context", "question"]
)

print("✅ Prompt template ready!")

✅ Prompt template ready!


## 🔗 Step 4 — Full RAG Chain

In [35]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
    """Combine retrieved chunks into a single context string with source info."""
    parts = []
    for i, doc in enumerate(retrieved_docs):
        source = doc.metadata.get("source", "unknown")
        page   = doc.metadata.get("page",   "")
        sheet  = doc.metadata.get("sheet",  "")
        label  = f"[Doc {i+1} | {os.path.basename(source)}"
        if page: label += f" | Page {page}"
        if sheet: label += f" | Sheet: {sheet}"
        label += "]"
        parts.append(f"{label}\n{doc.page_content}")
    return "\n\n".join(parts)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    "context":  retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

main_chain = parallel_chain | prompt | llm | parser

print("✅ RAG chain ready!")

✅ RAG chain ready!


In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define your prompt template
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question based on the following context:
    {context}

    Question: {question}
    """
)

# Define your main chain
main_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ── 1. Format retrieved docs into clean context text ──────────────
def format_docs(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs):
        source = doc.metadata.get("source", "unknown")
        page   = doc.metadata.get("page", "")
        sheet  = doc.metadata.get("sheet", "")
        label  = f"[Doc {i+1} | {os.path.basename(source)}"
        if page:  label += f" | Page {page}"
        if sheet: label += f" | Sheet: {sheet}"
        label += "]"
        parts.append(f"{label}\n{doc.page_content}")
    return "\n\n".join(parts)

# ── 2. Prompt that forces the LLM to use ONLY the context ─────────
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not found in the context, say:
"I don't have enough information in the provided documents."

Context:
{context}

Question: {question}
""")

# ── 3. Build the RAG chain correctly ──────────────────────────────
# More explicit chain with better prompting
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a document Q&A assistant.
Use ONLY the retrieved context to answer.
Always answer in detail from the context.
Context: {context}"""),
    ("human", "{question}")
])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test with a keyword that's DEFINITELY in your doc
answer = rag_chain.invoke("What PDF all about?")
print(answer)

The PDF appears to be about AI Operations (AI OPS) and discusses various topics related to artificial intelligence, natural language processing, and information retrieval. The context provided is from different pages of the PDF, including pages 223, 280, 50, and 294.

Page 223 discusses evaluating AI systems, specifically a system that extracts the current employer from a resume PDF. It highlights the importance of evaluating each component of the system independently to identify where it fails.

Page 280 mentions the term "document" and how it refers to both "document" and "chunk" in the context of classical NLP and information retrieval terminologies. It also touches on retrieval algorithms and their application in RAG (Retrieval-Augmented Generation).

Page 50 talks about the benefits of AI in processing and retrieving information from documents, such as contracts, disclosures, and papers. It also mentions the use of AI in summarizing websites, research, and creating reports.

Page 

## 💬 Ask Questions About Your Documents

---
## 📋 Quick Reference

| Component | Old (OpenAI) | New (HuggingFace) |
|---|---|---|
| API Key | `OPENAI_API_KEY` | `HUGGINGFACEHUB_API_TOKEN` |
| Embeddings | `OpenAIEmbeddings` | `HuggingFaceEmbeddings` |
| LLM | `ChatOpenAI (gpt-4o-mini)` | `HuggingFaceEndpoint (Mistral-7B)` |
| Vector Store | FAISS | FAISS (unchanged) |
| Document Input | YouTube transcript only | PDF + Excel + Word |

### 🔁 To switch models, change `repo_id`:
```python
repo_id = "google/flan-t5-large"               # Lightweight
repo_id = "HuggingFaceH4/zephyr-7b-beta"       # Chat-optimized  
repo_id = "mistralai/Mistral-7B-Instruct-v0.2" # Best quality (default)
```

### 📁 Supported file types:
- `PDF` → `PyPDFLoader` — extracts text page-by-page
- `Excel (.xlsx)` → `pandas` — reads all sheets
- `Word (.docx)` → `Docx2txtLoader` — extracts all paragraphs